In [ ]:
import pandas as pd

In [ ]:
from google.cloud import bigquery

client = bigquery.Client()

query = """
    SELECT *
    FROM `numeric-advice-452700-j9.neo_bank_.user_engagement`
"""

df_user_engagement = client.query(query).to_dataframe()

# Ahora df_user_engagement ya tiene todo listo, tal como salía del groupby:
print(df_user_engagement.head())

In [ ]:
# Asegúrate de que la fecha sea datetime
# df_user_engagement['create_date_notification'] = pd.to_datetime(df_user_engagement['create_date_notification'])

# Agrupa por usuario (suponiendo que hay una columna 'user_id')
user_engagement = df_user_engagement
user_engagement.columns

In [ ]:
from sklearn.preprocessing import MinMaxScaler
# Selecciona las columnas a normalizar
cols_to_scale = [
    'num_transactions', 'total_amount_usd', 'num_transaction_types',
    'num_merchants', 'num_countries', 'pct_card_present',
    'num_notifications', 'num_contacts', 'num_referrals', 'num_successful_referrals'
]
scaler = MinMaxScaler()
user_engagement_scaled = user_engagement.copy()
user_engagement_scaled[cols_to_scale] = scaler.fit_transform(user_engagement[cols_to_scale])
user_engagement_scaled

In [ ]:
user_engagement_scaled['engagement_score'] = (
    0.2 * user_engagement_scaled['num_transactions'] +
    0.15 * user_engagement_scaled['total_amount_usd'] +
    0.1 * user_engagement_scaled['num_transaction_types'] +
    0.1 * user_engagement_scaled['num_merchants'] +
    0.05 * user_engagement_scaled['num_countries'] +
    0.1 * user_engagement_scaled['pct_card_present'] +
    0.1 * user_engagement_scaled['num_notifications'] +
    0.05 * user_engagement_scaled['num_contacts'] +
    0.1 * user_engagement_scaled['num_successful_referrals']
)

In [ ]:
# Define thresholds
high_threshold = user_engagement_scaled['engagement_score'].quantile(0.80)
low_threshold = user_engagement_scaled['engagement_score'].quantile(0.40)
# Assign engagement levels
user_engagement_scaled['engagement_level'] = user_engagement_scaled['engagement_score'].apply(
    lambda x: 'high' if x >= high_threshold else
              'medium' if x >= low_threshold else
              'low')

In [ ]:
user_engagement_scaled

In [ ]:
user_engagement_scaled.columns

In [ ]:
user_engagement_scaled.info()

In [ ]:
import plotly.express as px

# Generar tabla de frecuencias como DataFrame robusto
engagement_counts = (
    user_engagement_scaled['engagement_level']
    .value_counts(dropna=False)
    .rename_axis('engagement_level')
    .reset_index(name='num_users')
    .sort_values('engagement_level')  # opcional: orden alfabético
)

# Revisar la tabla
print(engagement_counts.head(), engagement_counts['num_users'].sum())

# Graficar
fig = px.bar(
    engagement_counts,
    x='engagement_level',
    y='num_users',
    text='num_users',
    color='engagement_level',
    color_discrete_sequence=px.colors.qualitative.Set2,
    labels={
        'engagement_level': 'Nivel de Engagement',
        'num_users': 'Número de Usuarios'
    },
    title='Distribución de usuarios por nivel de engagement'
)

fig.update_traces(textposition='outside')
fig.update_layout(
    xaxis_title='Nivel de Engagement',
    yaxis_title='Número de Usuarios',
    showlegend=False
)

fig.show()


In [ ]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Columnas a escalar
cols_to_scale = [
    'num_transactions', 'total_amount_usd', 'num_transaction_types',
    'num_merchants', 'num_countries', 'pct_card_present',
    'num_notifications', 'num_contacts', 'num_successful_referrals'
]

# 2. Copia del dataset original
user_engagement_scaled = user_engagement.copy()

# 3. Escalar valores con MinMaxScaler
scaler = MinMaxScaler()
user_engagement_scaled[cols_to_scale] = scaler.fit_transform(user_engagement[cols_to_scale])

# 4. Calcular engagement_score ponderado
weights = {
    'num_transactions': 0.2,
    'total_amount_usd': 0.15,
    'num_transaction_types': 0.10,
    'num_merchants': 0.10,
    'num_countries': 0.05,
    'pct_card_present': 0.10,
    'num_notifications': 0.10,
    'num_contacts': 0.05,
    'num_successful_referrals': 0.10
}

user_engagement_scaled['engagement_score'] = sum(
    user_engagement_scaled[col] * weight for col, weight in weights.items()
)

# 5. Visualizar la distribución del score
plt.figure(figsize=(8, 4))
sns.histplot(user_engagement_scaled['engagement_score'], kde=True, bins=30)
plt.title("Distribución del Engagement Score")
plt.xlabel("Engagement Score")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()

# 6. Categorizar niveles de engagement usando percentiles
user_engagement_scaled['engagement_level'] = pd.qcut(
    user_engagement_scaled['engagement_score'],
    q=[0, 0.4, 0.8, 1.0],
    labels=['low', 'medium', 'high']
)

# 7. Contar usuarios por nivel
engagement_counts = (
    user_engagement_scaled['engagement_level']
    .value_counts(dropna=False)
    .rename_axis('engagement_level')
    .reset_index(name='num_users')
    .sort_values('engagement_level')  # Orden alfabético
)

# 8. Graficar con Plotly
fig = px.bar(
    engagement_counts,
    x='engagement_level',
    y='num_users',
    text='num_users',
    color='engagement_level',
    color_discrete_sequence=px.colors.qualitative.Set2,
    labels={
        'engagement_level': 'Nivel de Engagement',
        'num_users': 'Número de Usuarios'
    },
    title='Distribución de usuarios por nivel de engagement'
)

fig.update_traces(textposition='outside')
fig.update_layout(
    xaxis_title='Nivel de Engagement',
    yaxis_title='Número de Usuarios',
    showlegend=False
)

fig.show()


In [ ]:
query = """
    SELECT *
    FROM `numeric-advice-452700-j9.neo_bank_.churn`
"""

df_churn = client.query(query).to_dataframe()

# Ahora df_churn ya tiene todo listo, tal como salía del groupby:
print(df_churn.head())

In [ ]:
# Unir por user_id
df = df_user_engagement.copy()

In [ ]:
df

In [ ]:
from sklearn.preprocessing import MinMaxScaler

cols_to_scale = [
    'num_transactions', 'total_amount_usd', 'num_transaction_types',
    'num_merchants', 'num_countries', 'pct_card_present',
    'num_notifications', 'num_contacts', 'num_successful_referrals'
]

X = df[cols_to_scale]
y = df['churned']  # Asegúrate que sea binario: 1 = churned, 0 = activo

# Escalar
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# Modelo
model = GradientBoostingClassifier(random_state=42)
model.fit(X_scaled, y)

In [ ]:
# Extraer importancia de cada feature
importances = model.feature_importances_

# Normalizar para que sumen 1
weights = importances / importances.sum()

# Asociar a los nombres
weights_df = pd.DataFrame({'feature': cols_to_scale, 'weight': weights})
print(weights_df.sort_values(by='weight', ascending=False))


In [ ]:
import numpy as np

# Engagement score = sumatoria de feature * peso
engagement_score = np.dot(X_scaled, weights)

df['engagement_score_ml'] = engagement_score


In [ ]:
df['engagement_level_ml'] = pd.qcut(
    df['engagement_score_ml'],
    q=[0, 0.4, 0.8, 1.0],
    labels=['low', 'medium', 'high']
)


In [ ]:
import plotly.express as px

engagement_counts = (
    df['engagement_level_ml']
    .value_counts()
    .reset_index(name='num_users')
    .rename(columns={'index': 'engagement_level_ml'})
)

fig = px.bar(
    engagement_counts,
    x='engagement_level_ml',
    y='num_users',
    text='num_users',
    color='engagement_level_ml',
    title='Distribución de Engagement Score (Gradient Boosting)'
)
fig.show()


In [ ]:
df.columns

In [ ]:
cols_to_scale = [
    'num_transactions', 'total_amount_usd', 'num_transaction_types',
    'num_merchants', 'num_countries', 'pct_card_present',
    'num_notifications', 'num_contacts', 'num_successful_referrals'
]

# Separar X e y
X = df[cols_to_scale]
y = df['churned']

# Escalar
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Modelo
model = GradientBoostingClassifier(random_state=42)
model.fit(X_scaled, y)

# Calcular engagement score
weights = model.feature_importances_ / model.feature_importances_.sum()
df['engagement_score_ml'] = np.dot(X_scaled, weights)

# Categorizar en niveles
df['engagement_level_ml'] = pd.qcut(
    df['engagement_score_ml'],
    q=[0, 0.4, 0.8, 1.0],
    labels=['low', 'medium', 'high']
)

# Agrupar por canal
if 'channel' in df.columns:
    grouped = (
        df.groupby(['channel', 'engagement_level_ml'])
        .size()
        .reset_index(name='num_users')
    )

    fig = px.bar(
        grouped,
        x='channel',
        y='num_users',
        color='engagement_level_ml',
        text='num_users',
        barmode='stack',  # 👈 Cambia a 'group' si prefieres barras lado a lado
        title='Distribución de engagement por canal',
        color_discrete_sequence=px.colors.qualitative.Set2
    )
else:
    counts = (
        df['engagement_level_ml']
        .value_counts()
        .reset_index(name='num_users')
        .rename(columns={'index': 'engagement_level_ml'})
        .sort_values(by='engagement_level_ml')
    )

    fig = px.bar(
        counts,
        x='engagement_level_ml',
        y='num_users',
        text='num_users',
        color='engagement_level_ml',
        title='Distribución de engagement (sin agrupación)',
        color_discrete_sequence=px.colors.qualitative.Set2
    )
    fig.update_layout(showlegend=False)

fig.update_traces(textposition='outside')
fig.update_layout(
    xaxis_title='Canal' if 'channel' in df.columns else 'Nivel de Engagement',
    yaxis_title='Número de Usuarios'
)

fig.show()


In [ ]:
cols_to_scale = [
    'num_transactions', 'total_amount_usd', 'num_transaction_types',
    'num_merchants', 'num_countries', 'pct_card_present',
    'num_notifications', 'num_contacts', 'num_successful_referrals'
]

# Separar X e y
X = df[cols_to_scale]
y = df['churned']

# Escalar
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Modelo
model = GradientBoostingClassifier(random_state=42)
model.fit(X_scaled, y)

# Calcular engagement score
weights = model.feature_importances_ / model.feature_importances_.sum()
df['engagement_score_ml'] = np.dot(X_scaled, weights)

# Categorizar en niveles
df['engagement_level_ml'] = pd.qcut(
    df['engagement_score_ml'],
    q=[0, 0.4, 0.8, 1.0],
    labels=['low', 'medium', 'high']
)

# Agrupar por canal
if 'plan' in df.columns:
    grouped = (
        df.groupby(['plan', 'engagement_level_ml'])
        .size()
        .reset_index(name='num_users')
    )

    fig = px.bar(
        grouped,
        x='plan',
        y='num_users',
        color='engagement_level_ml',
        text='num_users',
        barmode='stack',  # 👈 Cambia a 'group' si prefieres barras lado a lado
        title='Distribución de engagement por canal',
        color_discrete_sequence=px.colors.qualitative.Set2
    )
else:
    counts = (
        df['engagement_level_ml']
        .value_counts()
        .reset_index(name='num_users')
        .rename(columns={'index': 'engagement_level_ml'})
        .sort_values(by='engagement_level_ml')
    )

    fig = px.bar(
        counts,
        x='engagement_level_ml',
        y='num_users',
        text='num_users',
        color='engagement_level_ml',
        title='Distribución de engagement (sin agrupación)',
        color_discrete_sequence=px.colors.qualitative.Set2
    )
    fig.update_layout(showlegend=False)

fig.update_traces(textposition='outside')
fig.update_layout(
    xaxis_title='Canal' if 'plan' in df.columns else 'Nivel de Engagement',
    yaxis_title='Número de Usuarios'
)

fig.show()


In [ ]:
cols_to_scale = [
    'num_transactions', 'total_amount_usd', 'num_transaction_types',
    'num_merchants', 'num_countries', 'pct_card_present',
    'num_notifications', 'num_contacts', 'num_successful_referrals'
]

# Separar X e y
X = df[cols_to_scale]
y = df['churned']

# Escalar
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Modelo
model = GradientBoostingClassifier(random_state=42)
model.fit(X_scaled, y)

# Calcular engagement score
weights = model.feature_importances_ / model.feature_importances_.sum()
df['engagement_score_ml'] = np.dot(X_scaled, weights)

# Categorizar en niveles
df['engagement_level_ml'] = pd.qcut(
    df['engagement_score_ml'],
    q=[0, 0.4, 0.8, 1.0],
    labels=['low', 'medium', 'high']
)

# Agrupar por canal
if 'age_group' in df.columns:
    grouped = (
        df.groupby(['age_group', 'engagement_level_ml'])
        .size()
        .reset_index(name='num_users')
    )

    fig = px.bar(
        grouped,
        x='age_group',
        y='num_users',
        color='engagement_level_ml',
        text='num_users',
        barmode='stack',  # 👈 Cambia a 'group' si prefieres barras lado a lado
        title='Distribución de engagement por canal',
        color_discrete_sequence=px.colors.qualitative.Set2
    )
else:
    counts = (
        df['engagement_level_ml']
        .value_counts()
        .reset_index(name='num_users')
        .rename(columns={'index': 'engagement_level_ml'})
        .sort_values(by='engagement_level_ml')
    )

    fig = px.bar(
        counts,
        x='engagement_level_ml',
        y='num_users',
        text='num_users',
        color='engagement_level_ml',
        title='Distribución de engagement (sin agrupación)',
        color_discrete_sequence=px.colors.qualitative.Set2
    )
    fig.update_layout(showlegend=False)

fig.update_traces(textposition='outside')
fig.update_layout(
    xaxis_title='Canal' if 'age_group' in df.columns else 'Nivel de Engagement',
    yaxis_title='Número de Usuarios'
)

fig.show()


In [ ]:
cols_to_scale = [
    'num_transactions', 'total_amount_usd', 'num_transaction_types',
    'num_merchants', 'num_countries', 'pct_card_present',
    'num_notifications', 'num_contacts', 'num_successful_referrals'
]

# Separar X e y
X = df[cols_to_scale]
y = df['churned']

# Escalar
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Modelo
model = GradientBoostingClassifier(random_state=42)
model.fit(X_scaled, y)

# Calcular engagement score
weights = model.feature_importances_ / model.feature_importances_.sum()
df['engagement_score_ml'] = np.dot(X_scaled, weights)

# Categorizar en niveles
df['engagement_level_ml'] = pd.qcut(
    df['engagement_score_ml'],
    q=[0, 0.4, 0.8, 1.0],
    labels=['low', 'medium', 'high']
)

# Agrupar por canal
if 'country_name' in df.columns:
    grouped = (
        df.groupby(['country_name', 'engagement_level_ml'])
        .size()
        .reset_index(name='num_users')
    )

    fig = px.bar(
        grouped,
        x='country_name',
        y='num_users',
        color='engagement_level_ml',
        text='num_users',
        barmode='stack',  # 👈 Cambia a 'group' si prefieres barras lado a lado
        title='Distribución de engagement por canal',
        color_discrete_sequence=px.colors.qualitative.Set2
    )
else:
    counts = (
        df['engagement_level_ml']
        .value_counts()
        .reset_index(name='num_users')
        .rename(columns={'index': 'engagement_level_ml'})
        .sort_values(by='engagement_level_ml')
    )

    fig = px.bar(
        counts,
        x='engagement_level_ml',
        y='num_users',
        text='num_users',
        color='engagement_level_ml',
        title='Distribución de engagement (sin agrupación)',
        color_discrete_sequence=px.colors.qualitative.Set2
    )
    fig.update_layout(showlegend=False)

fig.update_traces(textposition='outside')
fig.update_layout(
    xaxis_title='Canal' if 'country_name' in df.columns else 'Nivel de Engagement',
    yaxis_title='Número de Usuarios'
)

fig.show()
